<a href="https://colab.research.google.com/github/j-ranasinghe/ACL-injury-detection/blob/main/1_multimodalopenai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Multimodal RAG


In [2]:
# # Install libraries
# !pip install -q pymupdf
# !pip install -q tools
# !pip install -q langchain
# !pip install -q langchain_community
# !pip install -q faiss-cpu
# !pip install -q optimum
# !pip install -q gptqmodel
# # restart after installing

In [1]:
# PDF handling
import fitz

# Transformers & models
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, CLIPModel, CLIPProcessor

# Images & utilities
from PIL import Image
import io, base64
import os
import torch
import numpy as np

# LangChain
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# ML utils
from sklearn.metrics.pairwise import cosine_similarity

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [ ]:
# Load CLIP model + processor
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Set model to evaluation mode (no gradient computation)
clip_model.eval()

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIP ready. Zero money spent. 💸✨


In [ ]:
# Embedding functions
def embed_image(image_data):
    """
    Embed an image using CLIP.

    Args:
        image_data: Path to image or PIL Image.

    Returns:
        np.ndarray: Normalized embedding vector.
    """
    # Load image if path is given
    image = Image.open(image_data).convert("RGB") if isinstance(image_data, str) else image_data

    # Preprocess
    inputs = clip_processor(images=image, return_tensors="pt").to(device)

    # Extract features
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True)  # Normalize

    return features.squeeze().cpu().numpy()


def embed_text(text):
    """
    Embed text using CLIP.
    Args:
        text (str): Text to embed.
    Returns:
        np.ndarray: Normalized embedding vector.
    """
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    ).to(device)

    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True)  # Normalize

    return features.squeeze().cpu().numpy()


In [ ]:
## Process PDF
pdf_path="/content/Poster_.pdf"
doc=fitz.open(pdf_path)
# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {}  # Store actual image data for LLM

# Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [ ]:
for i,page in enumerate(doc):
    # process text
    text=page.get_text()
    if text.strip():
        # create temporary document for splitting
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])

        #Embed each chunk using CLIP
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)

    ## process images

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            # Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            # Create unique identifier
            image_id = f"page_{i}_img_{img_index}"

            # Store image as base64 for later use with GPT-4V
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64

            # Embed image using CLIP
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)

            # Create document for image
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)

        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()


In [ ]:
# Create unified FAISS vector store with CLIP embeddings
embeddings_array = np.array(all_embeddings)
embeddings_array

array([[ 0.03384745, -0.00801811,  0.02137916, ...,  0.00514414,
        -0.02241835, -0.07086808],
       [ 0.00820632,  0.01056555, -0.01861477, ...,  0.03698909,
         0.04513241, -0.02299471],
       [ 0.01699531, -0.01240848,  0.01353017, ...,  0.10178835,
         0.01013285, -0.04138317],
       ...,
       [-0.01558066, -0.01965371,  0.01017718, ...,  0.09937815,
         0.01808759,  0.0132741 ],
       [ 0.00866319,  0.00684648,  0.00072737, ...,  0.0391885 ,
         0.01835421, -0.04149631],
       [ 0.00193618,  0.01977539, -0.08579599, ...,  0.09856613,
         0.04977917, -0.05710507]], shape=(16, 512), dtype=float32)

In [ ]:
(all_docs,embeddings_array)

([Document(metadata={'page': 0, 'type': 'text'}, page_content='Question Answering in a Low-Resource Language: Dataset and\nDeep Learning Adaptations for Sinhala\nJanani Ranasinghe (janani.20210926@iit.ac.lk) and  Ruvan Weerasinghe \nSinhala as a Language\nSpoken by over 20 million people in Sri Lanka.\nIndo-Aryan language, written in its own script.\nResults and Discussion\nDataset and experimental setups\nAdamW is used as the optimizer for all models and a learning\nrate of 5e-6. The batch size was set at 8 and finetuned for 5\nepochs.'),
  Document(metadata={'page': 0, 'type': 'text'}, page_content='rate of 5e-6. The batch size was set at 8 and finetuned for 5\nepochs. \nThe experiments were conducted using the HuggingFace\nTransformers library .\nThe SQuAD v1.1 was translate in Sinhala using Google\ntranslate API.\nIntroduction\nThe sudden surge in research in the QA domain is largely\ndriven by the availability of publicly available annotated QA\ndatasets.\nThe main contributions o

In [ ]:
# Create custom FAISS index since we have precomputed embeddings
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None,  # We're using precomputed embeddings
    metadatas=[doc.metadata for doc in all_docs]
)
vector_store

In [ ]:
model_name = "TheBloke/guanaco-7B-HF"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

gen_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.7
)

# Test
print(gen_pipe("Hello! How are you?")[0]["generated_text"])



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Hello! How are you? I am a creative designer based in Tampa, Florida. I work in a variety of disciplines including graphic design, illustration, and creative direction.
I also have a strong interest in type design and lettering.
I am a big fan of good design and I try to apply that philosophy to every project I work on. I strive to create work that is engaging, thoughtful, and visually appealing.
I am always up for a new challenge or project, so if you have something interesting that you think I can help with, feel free to get in touch.
Let's talk about how we can work together!


In [ ]:
class LocalChatModel:
    def __init__(self, model, tokenizer, max_new_tokens=256, temperature=0.7):
        from transformers import pipeline
        self.generator = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=max_new_tokens,
            temperature=temperature
        )

    def __call__(self, messages):
        prompt = ""
        # from langchain.schema.messages import HumanMessage
        for msg in messages:
            if isinstance(msg, HumanMessage):
                content = msg.content
                # If content is a list of dicts
                if isinstance(content, list):
                    # Extract text from each dict
                    texts = []
                    for item in content:
                        if isinstance(item, dict):
                            # Common key is 'page_content', fallback to 'text'
                            text = item.get("page_content") or item.get("text") or str(item)
                            texts.append(text)
                        else:
                            texts.append(str(item))
                    prompt += " ".join(texts) + "\n"
                else:
                    prompt += str(content) + "\n"

        outputs = self.generator(prompt)
        return type("Response", (object,), {"content": outputs[0]['generated_text']})

    def invoke(self, messages):
        return self.__call__(messages)



# Initialize the local model
llm = LocalChatModel(model, tokenizer)


Device set to use cuda:0


In [ ]:
llm

In [ ]:
def retrieve_multimodal(query, k=5):
    """Unified retrieval using CLIP embeddings for both text and images."""
    # Embed query using CLIP
    query_embedding = embed_text(query)

    # Search in unified vector store
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )

    return results

In [ ]:
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for GPT-4V."""
    content = []

    # Add the query
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })

    # Separate text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]

    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })

    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })

    # Add instruction
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })

    return HumanMessage(content=content)

In [ ]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    # Retrieve relevant documents
    context_docs = retrieve_multimodal(query, k=5)

    # Create multimodal message
    message = create_multimodal_message(query, context_docs)

    # Get response from GPT-4V
    response = llm.invoke([message])

    # Print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")

    return response.content

In [ ]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "Summarize the main findings from the document"
    ]

    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)


Query: Summarize the main findings from the document
--------------------------------------------------

Retrieved 5 documents:
  - Text from page 0: performance to F1: 73.52%,
showing benefits of multilingual training [Best Performance]
LARGE
Questi...
  - Text from page 0: systems is the lack of publicly available annotated datasets
specifically tailored for Sinhala 
Unli...
  - Text from page 0: rate of 5e-6. The batch size was set at 8 and finetuned for 5
epochs. 
The experiments were conducte...
  - Text from page 0: Performance per
Question type
Conclusion
Clean Dataset
1.Remove Zero-width joiners, extra
spaces, un...
  - Text from page 0: Question Answering in a Low-Resource Language: Dataset and
Deep Learning Adaptations for Sinhala
Jan...


Answer: Question: Summarize the main findings from the document

Context:
 Text excerpts:
[Page 0]: performance to F1: 73.52%,
showing benefits of multilingual training [Best Performance]
LARGE
Question Type Impact: Models performed best o